# Mode-classification calibration on GS2 linear databases (Fusion_PhD-7k70, lqwx, use8)

Calibrates the cutoffs of `general_analysis.mode_classification` on every GS2 Latin-hypercube database with per-species fluxes.
Three indicators per case from the final time: tearing parameter `T` (Hatch 2012), signed frequency `omega` (ion direction > 0, pyrokinetics
convention) and `chi_e/chi_i` (Kotschenreuther 2019). Consensus: a label only if every indicator agrees, otherwise `mixed`.
**Fusion_PhD-lqwx (2026-10-01):** the user's final criteria (KBM: T<0.05, omega>0, 0.25<chi_e/chi_i<4; MTM: T>0.15, omega<0).
**Fusion_PhD-use8 (2026-10-05):** indicators now come from each database's cube (`pyroscan.nc`-style `cube.nc` + `pyroscan.json`) through
`mc.indicators(scan, ds)` (`scan.sample_pyro`, Fusion_PhD-k8w8); no run directory is read and no per-run `Pyro` is built. The D_e/chi_e cut and the
7k70 old->new transition table (which needed it) were dropped by the user and are gone with `de_chi`.

In [ ]:
import os, warnings
from multiprocessing import Pool
from pathlib import Path
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from dotenv import load_dotenv
from general_analysis import mode_classification as mc

warnings.filterwarnings("ignore")
ROOT = next(path for path in (Path.cwd(), *Path.cwd().parents) if (path / "pyproject.toml").is_file() and (path / "notebooks").is_dir())
load_dotenv(ROOT / "local.env")
data_root = Path(os.environ["GK_DATA_ROOT"])
code_, run_template, project = "GS2", "Runs", "LATIN_HYPERCUBE"
# database label: (case, scan_information)
# database label: (case, scan_information). Every GS2 LATIN_HYPERCUBE database; load failures are reported.
databases = {"NSTX_MTM n1000": ("NSTX_MTM", "beta_q_shat_ky_n1000"), "NSTX_MTM n300": ("NSTX_MTM", "beta_q_shat_ky_n300"),
             "SPR-045": ("SPR-045", "kbm_8d"), "M1": ("M1", "kbm_8d"), "R4": ("R4", "kbm_8d")}
databases |= {f"SPR-008 {s}": ("SPR-008", s) for s in ("aLTe_shat", "aLTe_q", "aLTe_aLn", "aLTe_aLn_shat", "aLTe_aLn_q_n200",
              "aLTe_aLn_q_n500", "aLTe_aLn_shat_q_n200", "aLTe_aLn_shat_q_n500")}
n_spot, seed = 5, 0
plot_dir = ROOT / "Plots" / "mode_classification_lqwx"; plot_dir.mkdir(parents=True, exist_ok=True)
plt.style.use(Path(mc.__file__).parent / "paper.mplstyle")

In [ ]:
rows, cubes = [], {}
for db, (case, scan_name) in databases.items():
    try:
        scan, ds = mc.load_gs2_cube(data_root / code_ / run_template / project / case / scan_name)
        ind = mc.indicators(scan, ds).to_dataframe()
        cubes[db] = ds
        gamma = ds["growth_rate"].squeeze().pint.dequantify().values
        rows.append(ind.assign(db=db, run=ind.sample_name, growth_rate=gamma))
    except Exception as e:
        rows.append(pd.DataFrame([dict(db=db, run="", err=repr(e)[:120])]))
raw = pd.concat(rows, ignore_index=True)

In [ ]:
# Nothing is dropped silently: load failures, then non-positive growth rate, then non-finite indicators.
failed = raw[raw.get("err").notna()] if "err" in raw else raw.iloc[:0]
ok = raw.drop(failed.index)
failed.to_csv(plot_dir / "load_failures.csv", index=False)
print(failed.groupby("db").err.first().to_dict())
print("loaded", raw.groupby("db").size().to_dict(), "| load failures", failed.groupby("db").size().to_dict())
stable = ok[ok.growth_rate <= 0]
ok = ok[ok.growth_rate > 0]
nonfinite = ok[~np.isfinite(ok[["T", "omega", "chi_ratio"]]).all(axis=1)]
df = ok.drop(nonfinite.index).copy()
print("growth_rate <= 0", stable.groupby("db").size().to_dict(), "| non-finite indicator (zero flux)", nonfinite.groupby("db").size().to_dict())
print("kept", df.groupby("db").size().to_dict())

## Consensus labels and single-indicator disagreement

In [ ]:
df["label"] = mc.classify(df)
frac = df.groupby("db").label.value_counts().unstack(fill_value=0).reindex(columns=["KBM", "MTM", "mixed"], fill_value=0)
counts = frac.assign(total=frac.sum(axis=1), loaded=raw.groupby("db").size(), load_failed=failed.groupby("db").size(),
                     stable=stable.groupby("db").size(), zero_flux=nonfinite.groupby("db").size()).fillna(0).astype(int)
print(counts); counts.to_csv(plot_dir / "counts.csv")
single = pd.DataFrame({
    "omega": np.where(df.omega > 0, "KBM", "MTM"),
    "T": np.select([df["T"] < mc.T_BAL, df["T"] > mc.T_TEAR], ["KBM", "MTM"], "mixed"),
    "chi": np.where(df.chi_ratio.between(*mc.CHI_KBM), "KBM", "mixed")})
rows = {k: df.assign(s=v).groupby("db").apply(lambda g: ((g.s != "mixed") & (g.s != g.label)).sum()) for k, v in single.items()}
print("cases where the single indicator gives KBM/MTM but the consensus label differs:")
print(pd.DataFrame(rows))


## Comparison with Kotschenreuther 2019, Table 1
Table 1 (A): MHD-like (KBM) chi_i/chi_e ~ 1; MTM chi_i/chi_e ~ 1/10. (B): ITG/TEM chi_e/chi_i 1/4 to 1.
Shown as chi_e/chi_i (inverse of the table's chi_i/chi_e). Boxes are the 25-75% range of our labelled cases.

In [ ]:
fig, ax = plt.subplots(figsize=(5, 4))
groups = [df[df.label == l].chi_ratio for l in ("KBM", "MTM", "mixed")]
ax.boxplot([g.clip(-50, 500) for g in groups], tick_labels=[f"{l}\n(n={len(g)})" for l, g in zip(("KBM", "MTM", "mixed"), groups)], showfliers=False)
ax.scatter([1, 2], [1, 10], marker="*", s=200, c="gold", edgecolor="k", label="Kotschenreuther T1", zorder=5)
ax.set_ylabel("chi_e/chi_i"); ax.set_yscale("log"); ax.legend()
fig.savefig(plot_dir / "gs2_chi_e_over_chi_i_per_mode_label.png")
print(df.groupby("label")[["chi_ratio", "T", "omega"]].quantile([.25, .5, .75]).round(3))